## Bound Entangled State 纯随机初始化 get_ansatz

In [ ]:
import torch
import numpy as np
import sys
import matplotlib.pyplot as plt
import seaborn as sns

# ================= 配置 =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64) 
np.set_printoptions(threshold=sys.maxsize, linewidth=200, precision=6, suppress=True)
print(f"Running on: {device}")
TASK_DIM = 7           # 量子比特维度 (2-10)
TASK_RANK = 4          # Ansatz秩
TASK_MAX_ATTEMPTS = 30 # 最大尝试次数

# ================= 函数 =================
def get_ansatz(dim, rank):           #构造参数矩阵A，维度为 (dim^2, rank)，元素为复数
    real = torch.randn(dim * dim, rank, device=device)
    imag = torch.randn(dim * dim, rank, device=device)
    A = torch.complex(real, imag)
    A.requires_grad = True
    return A

def get_rho(A):                # Cholesky decomposition构造密度矩阵
    rho_un = A @ A.mH
    trace = torch.trace(rho_un) + 1e-16 # 防止trace为0
    return rho_un / trace

def partial_transpose_torch(rho, dim):          # 计算部分转置，判断量子态是否为PPT态
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_pt = rho_reshaped.permute(0, 3, 2, 1).contiguous()
    return rho_pt.view(dim * dim, dim * dim)

def get_metrics(rho, dim):
    # PPT最小特征值
    pt = partial_transpose_torch(rho, dim)
    min_ppt = torch.min(torch.linalg.eigvalsh(pt))
    # 调整CCN范数
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_r = rho_reshaped.permute(0, 2, 1, 3).contiguous().view(dim*dim, dim*dim)
    r_norm = torch.linalg.matrix_norm(rho_r, ord='nuc')
    return min_ppt, r_norm

# ================= 修复临界NPT态为PPT态 =================
def try_repair_state(rho_np, dim):
    """
    尝试通过混合白噪声将微弱 NPT 的态修复为 PPT。
    rho_new = (1-p) * rho + p * (I/d^2)
    """
    print("   >>> 启动自动修复程序...")

    d2 = dim * dim
    I = np.eye(d2) / d2
    
    # 计算最小特征值
    rho_tensor = rho_np.reshape(dim, dim, dim, dim)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(d2, d2)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    
    if min_eig >= 0:
        return rho_np # 最小特征值大于等于0，不需要修复
    
    # 只要混合一点点白噪声，就能把负特征值拉正
    # p 的估算公式： p > |min_eig| * d^2 
    p_needed = abs(min_eig) * d2 * 1.5  # 多加50%余量以防万一
    
    print(f"   >>> 估算需要的白噪声比例 p = {p_needed:.8f}")
    
    # 尝试修复
    rho_repaired = (1 - p_needed) * rho_np + p_needed * I
    
    return rho_repaired

# ================= 生成束缚纠缠态 =================
def generate_be_robust(dim=TASK_DIM, rank=TASK_RANK, max_attempts=TASK_MAX_ATTEMPTS):
    print(f"\n 目标: 构造 Dim {dim}x{dim}, Rank {rank} ")
    
    for attempt in range(1, max_attempts + 1):
        A = get_ansatz(dim, rank)
        optimizer = torch.optim.Adam([A], lr=0.04)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=500)
        
        print(f"\n[Attempt {attempt}]")
        
        # 稍微放宽判定标准，捕捉“差一点点”的态
        best_candidate = None
        best_candidate_score = -1.0
        
        total_steps = 6000
        start_target = 1.01
        end_target = 1.0005 # 稍微高一点，留给修复用
        
        for i in range(total_steps):
            optimizer.zero_grad()
            rho = get_rho(A)
            min_pt, r_norm = get_metrics(rho, dim)
            
    
            # 如果 PPT 稍微负一点点 (>-0.001) 且 纠缠度还不错 (>1.002)
            # 存下来，最后尝试修复
            if min_pt.item() > -0.001 and r_norm.item() > 1.002:
                # 评分标准：PPT越接近0越好，Norm越大越好
                score = r_norm.item() - 100 * abs(min_pt.item())
                if score > best_candidate_score:
                    best_candidate_score = score
                    best_candidate = rho.detach().cpu().numpy()

            # --- 损失函数 ---
            decay = np.exp(-5.0 * i / total_steps)
            current_target = end_target + (start_target - end_target) * decay
            
            loss = 0
            ppt_weight = 1000.0 + (i / total_steps) * 9000.0
            
            if min_pt < 0:
                loss += torch.nn.functional.softplus(-min_pt * ppt_weight)
            else:
                loss += -min_pt * 0.1

            if r_norm < current_target:
                diff = current_target - r_norm
                ent_weight = 100.0 + (i / total_steps) * 400.0
                loss += diff * ent_weight
            
            loss += 0.0001 * torch.norm(A)
            
            loss.backward()
            optimizer.step()
            scheduler.step(loss)
            
            # 完美成功判定
            if min_pt.item() > -1e-8 and r_norm.item() > 1.00001:
                print(f"\n [PERFECT HIT] Step {i}")
                return rho.detach().cpu().numpy()
            
            if i % 1000 == 0:
                print(f"   Step {i:4d} | Target: {current_target:.5f} | PPT: {min_pt.item():.6f} | Norm: {r_norm.item():.6f}")
        
        # 循环结束，如果没有完美结果，尝试修复最佳候选者
        if best_candidate is not None:
            print("   >>> 尝试修复最佳候选者...")
            rho_fixed = try_repair_state(best_candidate, dim)
            
            # 验证修复结果
            rho_t = torch.tensor(rho_fixed, device=device)
            m_ppt, m_norm = get_metrics(rho_t, dim)
            
            print(f"   >>> 修复后: PPT={m_ppt.item():.8f}, Norm={m_norm.item():.8f}")
            
            if m_ppt.item() > -1e-15 and m_norm.item() > 1.0:
                print("🎉 [REPAIR SUCCESS] 成功修复为束缚纠缠态！")
                return rho_fixed
            else:
                print("   >>> 修复后纠缠度丢失，继续尝试...")
        else:
            print("   >>> 没有找到有修复价值的候选者。")

    print("\n All attempts failed.")
    return None

# ================= 运行 =================

rho_final = generate_be_robust(dim=TASK_DIM, rank=TASK_RANK, max_attempts=TASK_MAX_ATTEMPTS) 

if rho_final is not None:
    print("\n" + "="*60)
    print(" 最终结果 ")
    print("="*60)
    
    rho_np = rho_final
    
    # 打印矩阵供复制
    print("rho = np.array(")
    print(np.array2string(rho_np, separator=', '))
    print(")")
    
    # 最终严格物理验证
    dim = TASK_DIM
    rank = TASK_RANK
    
    # PPT 验证
    rho_ts = rho_np.reshape(dim, dim, dim, dim)
    rho_pt = rho_ts.transpose(0, 3, 2, 1).reshape(dim*dim, dim*dim)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    
    # 纠缠验证
    rho_r = rho_ts.transpose(0, 2, 1, 3).reshape(dim*dim, dim*dim)
    r_norm = np.sum(np.linalg.svd(rho_r, compute_uv=False))
    
    # Rank 验证
    s = np.linalg.svd(rho_np, compute_uv=False)
    # 因为修复过程加了满秩的白噪声，Rank 会变成 Full Rank (16)
    # 但白噪声极其微小 (1e-6级别)，所以主要成分的 Rank 还是 4
    # 我们看 "显著奇异值"
    eff_rank = np.sum(s > 1e-4) 
    
    print("-" * 30)
    print(f"1. PPT Check (Min Eig) : {min_eig:.9f}  [{'PASS' if min_eig > -1e-9 else 'FAIL'}]")
    print(f"2. Entanglement (Norm) : {r_norm:.9f}  [{'PASS' if r_norm > 1.0 else 'FAIL'}]")
    print(f"3. Significant Rank    : {eff_rank}          (Base Rank was {rank})")
    print("-" * 30)
    
    if min_eig > -1e-9 and r_norm > 1.0:
        print("结论: 这是一个有效的束缚纠缠态。")
    else:
        print("结论: 数据仍有瑕疵。")

# 确保 rho_final 是 numpy 格式
if isinstance(rho_final, torch.Tensor):
    rho_np = rho_final.detach().cpu().numpy()
else:
    rho_np = rho_final

# 文本打印
print("\n" + "="*50)
print("密度矩阵数值展示")
print("="*50)

# 设置打印选项
np.set_printoptions(threshold=np.inf, linewidth=200, precision=4, suppress=True)

# 打印实部
print("--- 实部 (Real Part) ---")
print(rho_np.real)

# 打印虚部
if np.max(np.abs(rho_np.imag)) > 1e-5:
    print("\n--- 虚部 (Imaginary Part) ---")
    print(rho_np.imag)
else:
    print("\n(虚部几乎为0，忽略不计)")


# 可视化 (Cityscape/Heatmap)
plt.figure(figsize=(12, 5))

# 实部热力图
plt.subplot(1, 2, 1)
sns.heatmap(rho_np.real, cmap="RdBu_r", center=0, square=True)
plt.title(f"Real Part (Dim=4, Rank=4 BE State)\nNorm={1.0049:.5f}")

# 虚部热力图
plt.subplot(1, 2, 2)
sns.heatmap(rho_np.imag, cmap="RdBu_r", center=0, square=True)
plt.title("Imaginary Part")

plt.tight_layout()
plt.show()

# 保存到文件

filename = f"../data/BE_States_Results/BE_State_{TASK_DIM}x{TASK_DIM}_Rank{TASK_RANK}_1.npy"
np.save(filename, rho_np)
print(f"\n 矩阵已保存至: {filename}")
print(f"读取方法: rho = np.load('{filename}')")

#  验证是否为 PPT (再次确认)

def check_ppt(rho):
    dim = int(np.sqrt(rho.shape[0]))
    rho_tensor = rho.reshape(dim, dim, dim, dim)
    # 对第二个子系统 (B) 进行转置 (索引 1, 3)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(dim*dim, dim*dim)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    return min_eig

ppt_val = check_ppt(rho_np)
print(f"\n 最终 PPT 检查: Min Eig = {ppt_val:.9f}")
if ppt_val > -1e-6:
    print("✅ 这是一个合格的 PPT 态。")
else:
    print("⚠️ 警告：仍有微小负特征值，可能需要更多白噪声混合。")



Running on: cpu

 目标: 构造 Dim 7x7, Rank 4 

[Attempt 1]


C:\Users\zyl\AppData\Roaming\Python\Python311\site-packages\torch\optim\lr_scheduler.py:1694: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  current = float(metrics)


   Step    0 | Target: 1.01000 | PPT: -0.126491 | Norm: 2.937191
   Step 1000 | Target: 1.00463 | PPT: -0.001179 | Norm: 1.005426
   Step 2000 | Target: 1.00229 | PPT: -0.000729 | Norm: 1.002748
   Step 3000 | Target: 1.00128 | PPT: -0.000228 | Norm: 1.001501
   Step 4000 | Target: 1.00084 | PPT: -0.000139 | Norm: 1.000945
   Step 5000 | Target: 1.00065 | PPT: -0.000097 | Norm: 1.000654
   >>> 尝试修复最佳候选者...
   >>> 启动自动修复程序...
   >>> 估算需要的白噪声比例 p = 0.02240598
   >>> 修复后: PPT=0.00015925, Norm=0.98287391
   >>> 修复后纠缠度丢失，继续尝试...

[Attempt 2]
   Step    0 | Target: 1.01000 | PPT: -0.123305 | Norm: 2.971768
   Step 1000 | Target: 1.00463 | PPT: -0.001120 | Norm: 1.005109
   Step 2000 | Target: 1.00229 | PPT: -0.000819 | Norm: 1.004387
   Step 3000 | Target: 1.00128 | PPT: -0.000119 | Norm: 1.001583
   Step 4000 | Target: 1.00084 | PPT: -0.000095 | Norm: 1.000961
   Step 5000 | Target: 1.00065 | PPT: -0.000047 | Norm: 1.000737
   >>> 尝试修复最佳候选者...
   >>> 启动自动修复程序...
   >>> 估算需要的白噪声比例 p = 0.0148

KeyboardInterrupt: 

In [ ]:
import numpy as np

# 假设你的矩阵已经命名为 rho (即你上面提供的代码)

# 1. 验证偏转置 (Partial Transpose, PPT判据)
dimA, dimB = 4, 4
rho_tensor = rho.reshape(dim, dim, dim, dim)

# 对子系统 B 求偏转置: 交换第二个和第四个索引 (即1和3)
rho_pt_tensor = rho_tensor.transpose((0, 3, 2, 1))
rho_pt = rho_pt_tensor.reshape((16, 16))

# 计算偏转置矩阵的特征值
eigenvalues_pt = np.linalg.eigvalsh(rho_pt)
min_eig = np.min(eigenvalues_pt)

print(f"偏转置后的最小特征值: {min_eig:.8f}")

TOLERANCE = -1e-8 # 容忍机器计算的数值误差
if min_eig < TOLERANCE:
    print("\n👉 【结论】：这不是束缚纠缠态！")
    print("该态具有负偏转置（NPT）。它是可提纯的【自由纠缠态】（Free Entangled）。")
else:
    print("该态具有正偏转置（PPT）。正在使用 CCNR 判据检测是否纠缠...")
    
    # 2. CCNR判据 (交叉范数重排判据)，用于检测 PPT 态是否真正纠缠
    R_tensor = rho_tensor.transpose((0, 2, 1, 3))
    R_matrix = R_tensor.reshape((16, 16))
    singular_values = np.linalg.svd(R_matrix, compute_uv=False)
    ccnr_value = np.sum(singular_values)
    
    print(f"CCNR 判据值 (Trace Norm): {ccnr_value:.6f}")
    
    if ccnr_value > 1.0 + TOLERANCE:
        print("\n👉 【结论】：确认！这是一个确凿的【两体束缚纠缠态】(Bound Entangled State)！")
        print("它不仅具有正偏转置(PPT)，同时被证明是不可分(纠缠)的。")
    else:
        print("\n👉 【结论】：它是PPT态，但CCNR判据值 <= 1。")
        print("这表示它内部的噪声可能过大，大概率已经退化成了【可分态】（Separable State）。如果要证明它是束缚纠缠，需要你使用当时跑代码时的专属纠缠目击者（Entanglement Witness）来验证。")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 生成 Horodecki 态 (d=3)
# ==========================================

def generate_horodecki_state_np(a=0.3):
    """
    根据 arXiv:1101.5754 生成 3x3 的 Horodecki PPT 纠缠态。
    这是纯 Numpy 实现，方便后续检验。
    """
    if not (0 <= a <= 1):
        raise ValueError("参数 a 必须在 [0, 1] 之间")
    
    b = (1 + a) / 2
    c = np.sqrt(1 - a**2) / 2
    
    rho = np.zeros((9, 9), dtype=complex)
    
    # 填充对角线
    rho[0,0]=a; rho[1,1]=a; rho[2,2]=a; rho[3,3]=a; rho[4,4]=a; rho[5,5]=a
    rho[6,6]=b; rho[7,7]=a; rho[8,8]=b
    
    # 填充非对角线
    rho[0,4]=a; rho[0,8]=a; rho[4,0]=a; rho[4,8]=a; rho[8,0]=a; rho[8,4]=a
    rho[6,8]=c; rho[8,6]=c
    
    # 归一化
    rho = rho / (8*a + 1)
    return rho

# [重要设定]：生成态并赋值给全局变量，供检验代码读取
target_dim = 3
# 我们选择 a=0.5，这是 Horodecki 态纠缠最强的参数点
rho_np = generate_horodecki_state_np(a=0.5) 

print("✅ Horodecki 态 (a=0.5, d=3) 已成功生成并加载至内存！\n")

# ==========================================
# 2. 你的检验程序 (完整保留你的逻辑)
# ==========================================

def check_bound_entanglement_in_notebook():
    # ================= 配置区 =================
    try:
        if 'rho_final' in globals() and rho_final is not None:
            rho_input = rho_final
        elif 'rho_np' in globals() and rho_np is not None:
            rho_input = rho_np
        else:
            print("❌ 错误：未找到生成的密度矩阵变量 (rho_final 或 rho_np)。")
            return

        if 'target_dim' in globals():
            d = target_dim
        else:
            d = int(np.sqrt(rho_input.shape[0]))
            print(f"⚠️ 未找到 target_dim 变量，自动推断维度为 d={d}")
            
    except Exception as e:
        print(f"❌ 读取数据时出错: {e}")
        return
    # ==========================================

    print(f"🔬 正在分析 {d}x{d} 维量子态 (Matrix: {d**2}x{d**2})...\n")

    # 1. 格式标准化 (Tensor -> Numpy, 厄米化, 归一化)
    if isinstance(rho_input, torch.Tensor):
        rho = rho_input.detach().cpu().numpy()
    else:
        rho = np.array(rho_input, copy=True)
    
    # 强行厄米化 (消除计算误差)
    rho = (rho + rho.conj().T) / 2
    # 强行归一化
    rho = rho / np.trace(rho)

    d2 = d * d
    tolerance = 1e-9 # 浮点数容差

    # ---------------------------------------------------
    # 2. 判据 A: PPT (Positive Partial Transpose)
    # ---------------------------------------------------
    rho_tensor = rho.reshape(d, d, d, d)
    # 转置子系统B (交换下标 1, 3)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(d2, d2)
    pt_evals = np.linalg.eigvalsh(rho_pt)
    min_pt_eig = np.min(pt_evals)

    is_ppt = min_pt_eig >= -tolerance

    # ---------------------------------------------------
    # 3. 判据 B: CCN (Computable Cross Norm / Realignment)
    # ---------------------------------------------------
    # 重排: indices (i, j, k, l) -> (i, k, j, l)
    rho_r = rho_tensor.transpose(0, 2, 1, 3).reshape(d2, d2)
    # 计算奇异值之和 (Trace Norm)
    singular_values = np.linalg.svd(rho_r, compute_uv=False)
    ccn_norm = np.sum(singular_values)

    is_entangled = ccn_norm > 1.0 + tolerance

    # ---------------------------------------------------
    # 4. 输出结果报告
    # ---------------------------------------------------
    print(f"1️⃣  [PPT 判据] 部分转置最小特征值: {min_pt_eig:.9f}")
    if is_ppt:
        print("    -> 结果: ✅ PPT (正部分转置)")
        print("    -> 含义: 该态不可被提纯。")
    else:
        print("    -> 结果: ❌ NPT (负部分转置)")
        print("    -> 含义: 该态是自由纠缠态 (Free Entangled)，不是束缚纠缠。")

    print(f"\n2️⃣  [CCN 判据] 重排范数 (Target > 1): {ccn_norm:.9f}")
    if is_entangled:
        print("    -> 结果: ✅ 确认纠缠")
    else:
        print("    -> 结果: ❓ 未检测到纠缠 (可能为可分态)")

    print("\n" + "="*40)
    if is_ppt and is_entangled:
        print("🏆 最终结论: 这是一个完美的【束缚纠缠态】! 🏆")
    elif not is_ppt:
        print("结论: 这是一个【自由纠缠态】 (NPT Entangled)。")
    else:
        print("结论: 无法确认纠缠 (可能是PPT可分态)。")
    print("="*40)

    # ---------------------------------------------------
    # 5. 可视化绘图
    # ---------------------------------------------------
    plt.figure(figsize=(12, 4))
    
    # 图1: 部分转置谱 (PT Eigenvalues)
    plt.subplot(1, 2, 1)
    # 给特征值排序后画图，看着更直观
    sorted_evals = np.sort(pt_evals)
    plt.plot(sorted_evals, 'b.-', markersize=8, label='PT Eigenvalues')
    plt.axhline(0, color='r', linestyle='--', linewidth=1.5, label='Zero Boundary')
    plt.title(f"PT Spectrum (Min Eig: {min_pt_eig:.1e})", fontsize=12)
    plt.xlabel("Eigenvalue Index")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True, alpha=0.4)
    
    # 图2: 密度矩阵实部热力图
    plt.subplot(1, 2, 2)
    # 对于 Horodecki 态，取一个合适的显示范围
    sns.heatmap(rho.real, cmap="RdBu_r", center=0, vmin=-0.15, vmax=0.15, 
                annot=True, fmt=".2f", annot_kws={"size": 7}) # 添加数字标注方便查看
    plt.title("Density Matrix Real Part (Horodecki a=0.5)", fontsize=12)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 3. 运行！
# ==========================================
check_bound_entanglement_in_notebook()

## 产生 Bound Entangled State 混合初始化 Hybrid Ansatz

In [ ]:
import torch
import numpy as np
import sys
import time
import random
import matplotlib.pyplot as plt
import seaborn as sns

# ================= 配置 =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64) 
np.set_printoptions(threshold=sys.maxsize, linewidth=200, precision=5, suppress=True)

print(f"Running on: {device}")

TASK_DIM = 5           # 量子比特维度 (2-10)
TASK_RANK = 4          # Ansatz秩
TASK_MAX_ATTEMPTS = 20 # 最大尝试次数

# ================= 基础函数 =================

def get_ansatz_hybrid(dim, rank):                 
    """
    混合初始化：
    不仅仅是随机，而是让初始态稍微接近 Mixed State (I/d)，
    这样离 PPT 区域更近，减少优化难度。
    """
    # 随机部分
    real = torch.randn(dim * dim, rank, device=device)
    imag = torch.randn(dim * dim, rank, device=device)
    A_rand = torch.complex(real, imag) # 随机复数矩阵
    
    # 单位矩阵部分 (Reshape 成列向量形式混合)
    # 这相当于给 ansatz 加一个 bias，使其不过于极端
    bias = torch.eye(dim * dim, rank, device=device) * 0.5
    
    A = A_rand + bias
    A.requires_grad = True
    return A

def get_rho(A):
    rho_un = A @ A.mH
    trace = torch.trace(rho_un) + 1e-16
    return rho_un / trace

def partial_transpose(rho, dim):
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_pt = rho_reshaped.permute(0, 3, 2, 1).contiguous()
    return rho_pt.view(dim * dim, dim * dim)

def get_metrics_high_dim(rho, dim):
    """
    [改进2] 针对高维的指标计算
    返回: 
    1. min_eig: 最小特征值 (用于最终判断)
    2. negativity_sum: 所有负特征值之和 (用于提供更强的梯度)
    3. r_norm: 重排范数
    """
    # PPT 相关
    pt = partial_transpose(rho, dim)
    eigvals = torch.linalg.eigvalsh(pt)
    
    min_eig = torch.min(eigvals)
    # 计算所有负特征值的平方和 (作为 Loss 梯度更平滑)
    # 或者直接用 sum(abs(negative_eigs))
    negatives = eigvals[eigvals < 0]
    negativity_loss = torch.sum(torch.square(negatives)) 
    
    # Entanglement 相关
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_r = rho_reshaped.permute(0, 2, 1, 3).contiguous().view(dim*dim, dim*dim)
    r_norm = torch.linalg.matrix_norm(rho_r, ord='nuc')
    
    return min_eig, negativity_loss, r_norm

# ================= 修复函数 =================
def try_repair_state(rho_np, dim):
    print("   >>> [Auto-Repair] 启动微量白噪声混合...")
    d2 = dim * dim
    I = np.eye(d2) / d2
    
    rho_tensor = rho_np.reshape(dim, dim, dim, dim)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(d2, d2)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    
    if min_eig >= 0: return rho_np
    
    # 动态计算需要的噪声比例
    p_needed = abs(min_eig) * d2 * 1.2 # 给 1.2 倍余量
    print(f"   >>> 需要混合 p = {p_needed:.8f}")
    
    rho_repaired = (1 - p_needed) * rho_np + p_needed * I
    return rho_repaired

# ================= 核心：高维优化器 =================
def set_seed(seed):
    """固定所有随机种子，确保可复现"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # 牺牲一点速度换取确定性（可选）
        # torch.backends.cudnn.deterministic = True
        # torch.backends.cudnn.benchmark = False

def solve_high_dim_optimized(dim=TASK_DIM, rank=TASK_RANK, max_attempts=TASK_MAX_ATTEMPTS, base_seed=42):
    print(f"\n🚀 [优化版] 启动高维攻坚: Dim {dim}x{dim}, Rank {rank}")
    print(f"🔧 策略: 动态两阶段优化 + 随机种子锁定 (Base Seed: {base_seed})")

    for attempt in range(1, max_attempts + 1):
        # 1. 设定当前尝试的种子
        current_seed = base_seed + attempt
        set_seed(current_seed)
        
        # 2. 初始化
        A = get_ansatz_hybrid(dim, rank)
        
        # 使用 AdamW
        optimizer = torch.optim.AdamW([A], lr=0.005, weight_decay=1e-5)
        
        # 增加步数，因为我们要进行精细微调
        total_steps = 20000 
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)
        
        print(f"\n[Attempt {attempt}/{max_attempts}] Seed: {current_seed}")
        
        best_candidate = None
        best_r_norm = 0
        best_min_eig = -1.0
        
        start_time = time.time()
        
        for i in range(total_steps):
            optimizer.zero_grad()
            rho = get_rho(A)
            
            # 计算指标
            # min_eig: PPT判据 (目标 >= 0)
            # neg_loss: 负特征值平方和 (优化动力)
            # r_norm: CCN判据 (目标 > 1)
            min_eig, neg_loss, r_norm = get_metrics_high_dim(rho, dim)
            
            # --- 记录最佳结果 ---
            # 条件：PPT 几乎满足 且 纠缠度高
            if min_eig.item() > -1e-5 and r_norm.item() > 1.0:
                # 只有当它是 PPT 且比之前的纠缠度更高时才保存
                if r_norm.item() > best_r_norm:
                    best_r_norm = r_norm.item()
                    best_min_eig = min_eig.item()
                    best_candidate = rho.detach().cpu().numpy()
            
            # --- 动态 Loss 设计 (核心优化) ---
            
            # 阶段判定：如果 PPT 已经合格 (>-1e-6)，则进入“贪婪纠缠模式”
            is_ppt_satisfied = min_eig.item() > -1e-6
            
            loss = 0.0
            
            if not is_ppt_satisfied:
                # [阶段一]：生存模式。首要任务是满足 PPT。
                # 强力惩罚负特征值，同时也给一点点奖励给 r_norm 防止坍缩成单位阵
                ppt_weight = 10000.0 * (1 + i/5000) # 随时间增加惩罚
                loss += neg_loss * ppt_weight
                loss += -r_norm * 0.1 # 弱引导
            else:
                # [阶段二]：贪婪模式。PPT 已满足，全力寻找更强的纠缠。
                # 只要 min_eig 还在安全区，就疯狂最大化 r_norm
                
                # 这里引入“安全边界”惩罚：虽然 > 0，但如果太靠近 0 也不好，稍微推远一点
                # 但主要权重给 r_norm
                loss += -r_norm * 50.0  # 强力推高 CCN
                
                # 保持 PPT 的约束 (Barrier Method)
                # 如果掉回负数，这个项会瞬间变大
                if min_eig < 0:
                    loss += neg_loss * 50000.0 
            
            # 反向传播
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            # --- 提前终止与日志 ---
            if i % 5000 == 0:
                print(f"   Step {i:5d} | PPT_Min: {min_eig.item():.7f} | Norm: {r_norm.item():.7f} | Mode: {'🟢Greedy' if is_ppt_satisfied else '🔴Survival'}")
            
            # 成功判定：非常严格的标准
            if min_eig.item() > -1e-9 and r_norm.item() > 1.0001:
                duration = time.time() - start_time
                print(f"\n🎉 [FOUND!] Step {i} (Seed {current_seed})")
                print(f"   耗时: {duration:.2f}s")
                print(f"   指标: PPT={min_eig.item():.9f}, CCN={r_norm.item():.9f}")
                return rho.detach().cpu().numpy()
        
        # 本次尝试结束，检查是否有幸存的“次优解”并尝试修复
        if best_candidate is not None:
            print(f"   ⚠️ 尝试修复最佳候选 (PPT={best_min_eig:.6f}, Norm={best_r_norm:.6f})...")
            rho_fixed = try_repair_state(best_candidate, dim)
            
            # 验证修复后的状态
            # 注意：修复通常会降低 Norm，所以必须再次检查
            rho_t = torch.tensor(rho_fixed, device=device)
            m_eig, _, m_norm = get_metrics_high_dim(rho_t, dim)
            
            if m_eig.item() > -1e-15 and m_norm.item() > 1.0:
                print(f"   ✨ 修复成功! Norm={m_norm.item():.6f}")
                return rho_fixed
            else:
                print(f"   💨 修复失败 (Norm降至 {m_norm.item():.6f})")

    print("\n💀 所有尝试均未找到完美符合条件的态。")
    return None

# ================= 运行 =================

target_dim =TASK_DIM
target_rank = TASK_RANK

rho_final = solve_high_dim_optimized(target_dim, target_rank, max_attempts=TASK_MAX_ATTEMPTS)

if rho_final is not None:
    print("\n" + "="*50)
    print("最终验证报告")
    print("="*50)
    
    rho_np = rho_final
    dim = target_dim
    
    # 1. PPT Check
    rho_ts = rho_np.reshape(dim, dim, dim, dim)
    rho_pt = rho_ts.transpose(0, 3, 2, 1).reshape(dim*dim, dim*dim)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    
    # 2. Entanglement Check
    rho_r = rho_ts.transpose(0, 2, 1, 3).reshape(dim*dim, dim*dim)
    r_norm = np.sum(np.linalg.svd(rho_r, compute_uv=False))
    
    # 3. Rank Check
    s = np.linalg.svd(rho_np, compute_uv=False)
    eff_rank = np.sum(s > 1e-4) # 稍微放宽阈值以过滤白噪声
    
    print(f"Dimension: {dim}x{dim}")
    print(f"PPT Min Eig: {min_eig:.9f}  [{'PASS' if min_eig > -1e-9 else 'FAIL'}]")
    print(f"Realignment: {r_norm:.9f}  [{'PASS' if r_norm > 1.0 else 'FAIL'}]")
    print(f"Approx Rank: {eff_rank}          (Target {target_rank})")
    
    if min_eig > -1e-9 and r_norm > 1.0:
        print("\n🏆 成功构造高维束缚纠缠态！")
        # 保存
        np.save(f"../data/TASK2_FILE_MAP/BE_State_d{dim}_r{target_rank}.npy", rho_np)
        print("矩阵已保存。")
    else:
        print("\n⚠️ 验证未完全通过。")

# ==========================================
# 1. 文本打印 (格式化输出)
# ==========================================
print("\n" + "="*50)
print("密度矩阵数值展示")
print("="*50)

# 设置打印选项：不省略，保留4位小数，一行显示更多
np.set_printoptions(threshold=np.inf, linewidth=200, precision=4, suppress=True)

# 打印实部
print("--- 实部 (Real Part) ---")
print(rho_np.real)

# 打印虚部 (如果有显著值)
if np.max(np.abs(rho_np.imag)) > 1e-5:
    print("\n--- 虚部 (Imaginary Part) ---")
    print(rho_np.imag)
else:
    print("\n(虚部几乎为0，忽略不计)")

# ==========================================
# 2. 可视化 (Cityscape/Heatmap)
# ==========================================
plt.figure(figsize=(12, 5))

# 实部热力图
plt.subplot(1, 2, 1)
sns.heatmap(rho_np.real, cmap="RdBu_r", center=0, square=True)
plt.title(f"Real Part (BE State)\nNorm={1.0049:.5f}")

# 虚部热力图
plt.subplot(1, 2, 2)
sns.heatmap(rho_np.imag, cmap="RdBu_r", center=0, square=True)
plt.title("Imaginary Part")

plt.tight_layout()
plt.show()

# ==========================================
# 3. 保存到文件
# ==========================================
filename = f"../data/BE_States_Results/BE_State_{TASK_DIM}x{TASK_DIM}_Rank{TASK_RANK}.npy"
np.save(filename, rho_np)
print(f"\n 矩阵已保存至: {filename}")
print(f"读取方法: rho = np.load('{filename}')")

# ==========================================
# 4. 验证是否为 PPT (再次确认)
# ==========================================
def check_ppt(rho):
    dim = int(np.sqrt(rho.shape[0]))
    rho_tensor = rho.reshape(dim, dim, dim, dim)
    # 对第二个子系统 (B) 进行转置 (索引 1, 3)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(dim*dim, dim*dim)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    return min_eig

ppt_val = check_ppt(rho_np)
print(f"\n 最终 PPT 检查: Min Eig = {ppt_val:.9f}")
if ppt_val > -1e-6:
    print("✅ 这是一个合格的 PPT 态。")
else:
    print("⚠️ 警告：仍有微小负特征值，可能需要更多白噪声混合。")

## 批量生成

In [ ]:
import torch
import numpy as np
import sys
import time
import random
import matplotlib.pyplot as plt
import seaborn as sns
import os  # 新增：用于文件路径管理

# ================= 配置与环境 =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64) 
np.set_printoptions(threshold=sys.maxsize, linewidth=200, precision=5, suppress=True)

print(f"Running on: {device}")

# ==========================================
#              1. 批量任务配置
# ==========================================
# 在这里输入你想跑的所有维度
TARGET_DIMS = [4]  
# 在这里输入你想跑的所有Rank
TARGET_RANKS = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16] 

# 输出文件夹名称
OUTPUT_DIR = "../data/BE_States_Results"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ================= 基础函数 (保持不变) =================

def get_ansatz_hybrid(dim, rank):
    real = torch.randn(dim * dim, rank, device=device)
    imag = torch.randn(dim * dim, rank, device=device)
    A_rand = torch.complex(real, imag)
    bias = torch.eye(dim * dim, rank, device=device) * 0.5
    A = A_rand + bias
    A.requires_grad = True
    return A

def get_rho(A):
    rho_un = A @ A.mH
    trace = torch.trace(rho_un) + 1e-16
    return rho_un / trace

def partial_transpose(rho, dim):
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_pt = rho_reshaped.permute(0, 3, 2, 1).contiguous()
    return rho_pt.view(dim * dim, dim * dim)

def get_metrics_high_dim(rho, dim):
    pt = partial_transpose(rho, dim)
    eigvals = torch.linalg.eigvalsh(pt)
    min_eig = torch.min(eigvals)
    negatives = eigvals[eigvals < 0]
    negativity_loss = torch.sum(torch.square(negatives)) 
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_r = rho_reshaped.permute(0, 2, 1, 3).contiguous().view(dim*dim, dim*dim)
    r_norm = torch.linalg.matrix_norm(rho_r, ord='nuc')
    return min_eig, negativity_loss, r_norm

def try_repair_state(rho_np, dim):
    d2 = dim * dim
    I = np.eye(d2) / d2
    rho_tensor = rho_np.reshape(dim, dim, dim, dim)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(d2, d2)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    
    if min_eig >= 0: return rho_np
    
    p_needed = abs(min_eig) * d2 * 1.2 
    # 限制 p 的最大值，防止为了修复而把纠缠全毁了
    if p_needed > 0.1: return rho_np 
    
    rho_repaired = (1 - p_needed) * rho_np + p_needed * I
    return rho_repaired

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# ================= 核心优化器 (封装版) =================

def solve_high_dim_optimized(dim, rank, max_attempts=30, base_seed=42):
    """
    针对特定 dim 和 rank 运行优化
    """
    print(f"\n🚀 [开始任务] Dim: {dim}x{dim}, Rank: {rank}")
    
    for attempt in range(1, max_attempts + 1):
        current_seed = base_seed + attempt * 100 # 种子间隔大一点
        set_seed(current_seed)
        
        A = get_ansatz_hybrid(dim, rank)
        optimizer = torch.optim.AdamW([A], lr=0.005, weight_decay=1e-5)
        
        # 针对不同维度动态调整步数
        total_steps = 20000 + (dim - 4) * 2000 
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)
        
        best_candidate = None
        best_r_norm = 0
        best_min_eig = -1.0
        
        # 简化日志输出，只打印关键信息
        # print(f"   Attempt {attempt} (Seed {current_seed})...", end="\r")
        
        for i in range(total_steps):
            optimizer.zero_grad()
            rho = get_rho(A)
            min_eig, neg_loss, r_norm = get_metrics_high_dim(rho, dim)
            
            if min_eig.item() > -1e-5 and r_norm.item() > 1.0:
                if r_norm.item() > best_r_norm:
                    best_r_norm = r_norm.item()
                    best_min_eig = min_eig.item()
                    best_candidate = rho.detach().cpu().numpy()
            
            is_ppt_satisfied = min_eig.item() > -1e-6
            loss = 0.0
            
            if not is_ppt_satisfied:
                ppt_weight = 10000.0 * (1 + i/5000)
                loss += neg_loss * ppt_weight
                loss += -r_norm * 0.1
            else:
                loss += -r_norm * 50.0
                if min_eig < 0:
                    loss += neg_loss * 50000.0 
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            if min_eig.item() > -1e-9 and r_norm.item() > 1.0001:
                print(f"   🎉 Attempt {attempt}: Found! PPT={min_eig.item():.2e}, CCN={r_norm.item():.5f}")
                return rho.detach().cpu().numpy()
        
        # 尝试修复
        if best_candidate is not None:
            rho_fixed = try_repair_state(best_candidate, dim)
            rho_t = torch.tensor(rho_fixed, device=device)
            m_eig, _, m_norm = get_metrics_high_dim(rho_t, dim)
            
            if m_eig.item() > -1e-15 and m_norm.item() > 1.0:
                print(f"   ✨ Attempt {attempt}: Repaired! PPT={m_eig.item():.2e}, CCN={m_norm.item():.5f}")
                return rho_fixed
    
    print(f"   ❌ Rank {rank} 尝试 {max_attempts} 次后未找到。")
    return None

# ================= 结果处理与保存函数 =================

def save_result(rho_np, dim, rank):
    """保存数据和图片"""
    base_name = f"BE_State_d{dim}_rank{rank}"
    
    # 1. 保存 NPY
    npy_path = os.path.join(OUTPUT_DIR, f"{base_name}.npy")
    np.save(npy_path, rho_np)
    
    # 2. 绘制并保存图片
    plt.figure(figsize=(12, 5))
    
    # 实部
    plt.subplot(1, 2, 1)
    sns.heatmap(rho_np.real, cmap="RdBu_r", center=0, square=True)
    
    # 计算 CCN 放在标题里
    rho_ts = rho_np.reshape(dim, dim, dim, dim)
    rho_r = rho_ts.transpose(0, 2, 1, 3).reshape(dim*dim, dim*dim)
    r_norm = np.sum(np.linalg.svd(rho_r, compute_uv=False))
    
    plt.title(f"Real Part (d={dim}, r={rank})\nCCN Norm={r_norm:.5f}")

    # 虚部
    plt.subplot(1, 2, 2)
    sns.heatmap(rho_np.imag, cmap="RdBu_r", center=0, square=True)
    plt.title("Imaginary Part")

    plt.tight_layout()
    img_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    plt.savefig(img_path, dpi=150)
    plt.close() # 这一步很重要，批量跑防止内存溢出
    
    print(f"   💾 Saved: {npy_path}")
    print(f"   🖼️ Saved: {img_path}")

# ================= 主执行循环 =================

def main():
    print(f"📁 结果将保存在文件夹: ./{OUTPUT_DIR}/")
    print(f"📋 计划任务:")
    print(f"   Dimensions: {TARGET_DIMS}")
    print(f"   Ranks:      {TARGET_RANKS}")
    
    success_count = 0
    total_count = 0
    
    start_time_all = time.time()

    # 双重循环：遍历所有 Dimension 和 Rank 的组合
    for d in TARGET_DIMS:
        for r in TARGET_RANKS:
            total_count += 1
            
            # 运行优化器
            # max_attempts 可以根据需要调小一点以节省时间，或者调大以提高成功率
            rho = solve_high_dim_optimized(d, r, max_attempts=10)
            
            if rho is not None:
                save_result(rho, d, r)
                success_count += 1
            else:
                print(f"   ⚠️ 跳过 d={d}, r={r} (未找到)")
                
    total_time = time.time() - start_time_all
    print("\n" + "="*50)
    print(f"🏁 批量生成结束!")
    print(f"⏱️ 总耗时: {total_time:.2f} 秒")
    print(f"✅ 成功率: {success_count}/{total_count}")
    print(f"📁 所有文件已保存在: ./{OUTPUT_DIR}")
    print("="*50)

if __name__ == "__main__":
    main()

## 检测

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

def check_bound_entanglement_in_notebook():
    # ================= 配置区 =================
    # 尝试从全局变量中获取数据，如果变量名不同，请手动修改这里
    try:
        # 优先使用 rho_final，如果它是None或不存在，尝试使用 rho_np
        if 'rho_final' in globals() and rho_final is not None:
            rho_input = rho_final
        elif 'rho_np' in globals() and rho_np is not None:
            rho_input = rho_np
        else:
            print("❌ 错误：未找到生成的密度矩阵变量 (rho_final 或 rho_np)。请先运行生成代码。")
            return

        # 获取维度
        if 'target_dim' in globals():
            d = target_dim
        else:
            # 尝试自动推断维度
            d = int(np.sqrt(rho_input.shape[0]))
            print(f"⚠️ 未找到 target_dim 变量，自动推断维度为 d={d}")
            
    except Exception as e:
        print(f"❌ 读取数据时出错: {e}")
        return
    # ==========================================

    print(f"🔬 正在分析 {d}x{d} 维量子态 (Matrix: {d**2}x{d**2})...\n")

    # 1. 格式标准化 (Tensor -> Numpy, 厄米化, 归一化)
    if isinstance(rho_input, torch.Tensor):
        rho = rho_input.detach().cpu().numpy()
    else:
        rho = np.array(rho_input, copy=True)
    
    # 强行厄米化 (消除计算误差)
    rho = (rho + rho.conj().T) / 2
    # 强行归一化
    rho = rho / np.trace(rho)

    d2 = d * d
    tolerance = 1e-9 # 浮点数容差

    # ---------------------------------------------------
    # 2. 判据 A: PPT (Positive Partial Transpose)
    # ---------------------------------------------------
    # 逻辑: 如果 min_eig < 0，则是NPT(自由纠缠)。如果 >= 0，则是PPT(可能是束缚纠缠)。
    rho_tensor = rho.reshape(d, d, d, d)
    # 转置子系统B (交换下标 1, 3)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(d2, d2)
    pt_evals = np.linalg.eigvalsh(rho_pt)
    min_pt_eig = np.min(pt_evals)

    is_ppt = min_pt_eig >= -tolerance

    # ---------------------------------------------------
    # 3. 判据 B: CCN (Computable Cross Norm / Realignment)
    # ---------------------------------------------------
    # 逻辑: 如果 CCN > 1，则一定是纠缠的。
    # 重排: indices (i, j, k, l) -> (i, k, j, l)
    rho_r = rho_tensor.transpose(0, 2, 1, 3).reshape(d2, d2)
    # 计算奇异值之和 (Trace Norm)
    singular_values = np.linalg.svd(rho_r, compute_uv=False)
    ccn_norm = np.sum(singular_values)

    is_entangled = ccn_norm > 1.0 + tolerance

    # ---------------------------------------------------
    # 4. 输出结果报告
    # ---------------------------------------------------
    print(f"1️⃣  [PPT 判据] 部分转置最小特征值: {min_pt_eig:.9f}")
    if is_ppt:
        print("    -> 结果: ✅ PPT (正部分转置)")
        print("    -> 含义: 该态不可被提纯。")
    else:
        print("    -> 结果: ❌ NPT (负部分转置)")
        print("    -> 含义: 该态是自由纠缠态 (Free Entangled)，不是束缚纠缠。")

    print(f"\n2️⃣  [CCN 判据] 重排范数 (Target > 1): {ccn_norm:.9f}")
    if is_entangled:
        print("    -> 结果: ✅ 确认纠缠")
    else:
        print("    -> 结果: ❓ 未检测到纠缠 (可能为可分态)")

    print("\n" + "="*40)
    if is_ppt and is_entangled:
        print("🏆 最终结论: 这是一个【束缚纠缠态】! 🏆")
    elif not is_ppt:
        print("结论: 这是一个【自由纠缠态】 (NPT Entangled)。")
    else:
        print("结论: 无法确认纠缠 (可能是PPT可分态)。")
    print("="*40)

    # ---------------------------------------------------
    # 5. 可视化绘图
    # ---------------------------------------------------
    plt.figure(figsize=(12, 4))
    
    # 图1: 部分转置谱 (PT Eigenvalues)
    plt.subplot(1, 2, 1)
    plt.plot(pt_evals, 'b.-', markersize=3, label='Eigenvalues')
    plt.axhline(0, color='r', linestyle='--', linewidth=1, label='Zero')
    plt.title(f"PT Spectrum (Min: {min_pt_eig:.1e})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    if min_pt_eig < 0:
        # 如果有负值，放大显示负值区域
        plt.ylim(min_pt_eig * 1.5, abs(min_pt_eig) * 1.5)
        plt.title(f"PT Spectrum (Zoomed on Negative)")

    # 图2: 密度矩阵实部
    plt.subplot(1, 2, 2)
    # 为了显示清晰，取对数或者限制范围
    sns.heatmap(rho.real, cmap="RdBu_r", center=0, vmin=-0.01, vmax=0.01)
    plt.title("Density Matrix (Real Part)")
    
    plt.tight_layout()
    plt.show()

# 运行检验
check_bound_entanglement_in_notebook()

In [ ]:
import numpy as np

def check_bound_entanglement(rho, dimA=None, dimB=None, tol=1e-8):
    """
    判断一个密度矩阵是否为两体束缚纠缠态 (Bipartite Bound Entangled State)
    """
    print("="*50)
    print("🔬 量子态束缚纠缠检测程序启动...")
    print("="*50)

    # 1. 维度推断
    dim = rho.shape[0]
    if dimA is None or dimB is None:
        dimA = int(np.sqrt(dim))
        dimB = dim // dimA
        if dimA * dimB != dim:
            raise ValueError("矩阵维度无法自动分解为两个子系统，请手动指定 dimA 和 dimB。")
    print(f"✅ 系统维度: {dim}x{dim} (推断为子系统A: {dimA}, 子系统B: {dimB})")

    # 2. 合法性检查 (迹为1，半正定)
    trace = np.trace(rho)
    if abs(trace - 1.0) > tol:
        print(f"⚠️ 警告: 矩阵的迹为 {trace.real:.6f}，不严格等于 1，但程序将继续计算。")
    
    eigenvalues_rho = np.linalg.eigvalsh(rho)
    if np.min(eigenvalues_rho) < -tol:
        print(f"❌ 错误: 矩阵存在负特征值 {np.min(eigenvalues_rho):.6e}，不是合法的量子态！")
        return

    # 3. 计算偏转置 (Partial Transpose)
    # 将矩阵 reshape 为4阶张量 (A_row, B_row, A_col, B_col)
    rho_tensor = rho.reshape((dimA, dimB, dimA, dimB))
    # 对 B 系统进行偏转置 (交换 B_row 和 B_col，即索引 1 和 3)
    rho_pt_tensor = rho_tensor.transpose((0, 3, 2, 1))
    rho_pt = rho_pt_tensor.reshape((dim, dim))

    # 计算偏转置后的特征值
    eigenvalues_pt = np.linalg.eigvalsh(rho_pt)
    min_eig_pt = np.min(eigenvalues_pt)
    print(f"📊 偏转置矩阵最小特征值: {min_eig_pt:.8e}")

    # 4. 判断 PPT (正偏转置) 还是 NPT (负偏转置)
    is_ppt = min_eig_pt >= -tol

    if not is_ppt:
        print("\n🚫 【最终结论】：它不是束缚纠缠态！")
        print("原因：该态具有负偏转置 (NPT)。这说明它是可以被直接提纯的【自由纠缠态】(Free Entangled State)。")
        return

    print("✅ 该态具有正偏转置 (PPT)。它满足束缚纠缠的第一个条件。")
    print("⏳ 正在计算 CCNR 判据 (交叉范数重排) 以检测是否存在纠缠...")

    # 5. 计算 CCNR 判据 (检测 PPT 态是否真正纠缠)
    # 重排操作 (Realignment): 交换 A_col 和 B_row (即索引 1 和 2)
    R_tensor = rho_tensor.transpose((0, 2, 1, 3))
    R_matrix = R_tensor.reshape((dimA * dimA, dimB * dimB))
    
    # 计算奇异值并求和 (迹范数)
    singular_values = np.linalg.svd(R_matrix, compute_uv=False)
    ccnr_value = np.sum(singular_values)
    print(f"📊 CCNR 判据值 (Trace Norm): {ccnr_value:.6f} (阈值为 1.0)")

    # 6. 综合得出最终结论
    if ccnr_value > 1.0 + tol:
        print("\n🎉 【最终结论】：绝对确认！这是一个【两体束缚纠缠态】(Bound Entangled State)！")
        print("原因：它同时满足 PPT（无法提纯）且 CCNR > 1（证明存在纠缠）。")
    else:
        print("\n⚠️ 【最终结论】：目前证据不足以证明它是束缚纠缠态。")
        print("原因：它是 PPT 态，但 CCNR 判据值 <= 1。这意味着它大概率已经退化成了无纠缠的【可分态】(Separable State)。")
        print("注：如果它是你在论文中看到的特殊构造态，CCNR判据可能失效，你需要借助针对该态专门设计的纠缠目击者(Entanglement Witness)矩阵来做最终判定。")


# ==========================================
# 在这里输入你的量子态矩阵 rho
# ==========================================
if __name__ == "__main__":
    # 请把你跑出的那段 np.array([...]) 直接粘贴在下面：
    rho = np.array(
    [[ 0.110717+0.j      , -0.042147-0.002136j, -0.03871 -0.088431j, -0.032333-0.006965j,  0.005818+0.035897j, -0.008078-0.007123j,  0.019954-0.027693j, -0.000799-0.016466j, -0.098119-0.030357j,
       0.032975-0.00606j , -0.016468+0.086672j,  0.015002+0.016456j, -0.030252-0.020544j,  0.011294+0.015986j, -0.002121+0.029806j,  0.009477+0.006134j],
     [-0.042147+0.002136j,  0.031656+0.j      ,  0.042462+0.04377j ,  0.021137+0.006982j,  0.005733-0.008536j,  0.00244 +0.010509j, -0.008862+0.007949j, -0.002983+0.004996j,  0.007846+0.01297j ,
      -0.023777-0.011179j, -0.017894-0.024044j, -0.009242-0.009113j,  0.022548+0.006461j, -0.00949 -0.005239j, -0.005817-0.029118j, -0.007208-0.006885j],
     [-0.03871 +0.088431j,  0.042462-0.04377j ,  0.156452+0.j      ,  0.044196-0.020551j, -0.033486+0.001779j,  0.006139+0.003039j, -0.001437+0.03936j ,  0.004251+0.006756j,  0.049766-0.029523j,
      -0.028067+0.028089j, -0.100008-0.001044j, -0.041196+0.013788j,  0.029473-0.023898j, -0.029068+0.003006j, -0.056925-0.041351j, -0.018943-0.005679j],
     [-0.032333+0.006965j,  0.021137-0.006982j,  0.044196+0.020551j,  0.020448+0.j      , -0.006443-0.00624j ,  0.00179 +0.003445j, -0.01059 +0.016275j, -0.002111+0.00581j ,  0.032413+0.018141j,
      -0.016363+0.0061j  , -0.012304-0.008907j, -0.014615+0.000503j,  0.009379+0.001497j, -0.00901 -0.004067j, -0.015402-0.018162j, -0.007477-0.005157j],
     [ 0.005818-0.035897j,  0.005733+0.008536j, -0.033486-0.001779j, -0.006443+0.00624j ,  0.059896+0.j      ,  0.010659+0.019225j,  0.005186-0.031471j, -0.010961-0.000253j, -0.077861+0.013081j,
      -0.019386-0.036408j,  0.032712+0.040006j,  0.028404+0.000193j,  0.020798+0.00672j ,  0.010087+0.006372j,  0.006866-0.007065j, -0.004596+0.000088j],
     [-0.008078+0.007123j,  0.00244 -0.010509j,  0.006139-0.003039j,  0.00179 -0.003445j,  0.010659-0.019225j,  0.012741+0.j      , -0.005644-0.007965j, -0.001189+0.003094j, -0.009329+0.02099j ,
      -0.018997+0.006395j,  0.00656 +0.007561j,  0.002035-0.004031j,  0.009555-0.013215j,  0.003405+0.001691j, -0.00866 -0.003658j, -0.002563+0.003349j],
     [ 0.019954+0.027693j, -0.008862-0.007949j, -0.001437-0.03936j , -0.01059 -0.016275j,  0.005186+0.031471j, -0.005644+0.007965j,  0.036846+0.j      ,  0.007397-0.005446j, -0.01548 -0.080839j,
       0.022643-0.010542j, -0.025727+0.00429j ,  0.01078 +0.015412j,  0.007845+0.006381j, -0.003943+0.013037j,  0.004216+0.016514j, -0.001045+0.006295j],
     [-0.000799+0.016466j, -0.002983-0.004996j,  0.004251-0.006756j, -0.002111-0.00581j , -0.010961+0.000253j, -0.001189-0.003094j,  0.007397+0.005446j,  0.005519+0.j      ,  0.012362-0.019739j,
       0.006427+0.006472j, -0.008458-0.01634j , -0.001182-0.000863j, -0.000061-0.002326j, -0.002384+0.00147j ,  0.000331+0.007023j,  0.001039+0.003455j],
     [-0.098119+0.030357j,  0.007846-0.01297j ,  0.049766+0.029523j,  0.032413-0.018141j, -0.077861-0.013081j, -0.009329-0.02099j , -0.01548 +0.080839j,  0.012362+0.019739j,  0.242803+0.j      ,
       0.01867 +0.069645j,  0.023206-0.088123j, -0.041079+0.01863j , -0.02004 +0.023296j, -0.019002-0.017555j, -0.017364+0.013416j, -0.007849-0.00191j ],
     [ 0.032975+0.00606j , -0.023777+0.011179j, -0.028067-0.028089j, -0.016363-0.0061j  , -0.019386+0.036408j, -0.018997-0.006395j,  0.022643+0.010542j,  0.006427-0.006472j,  0.01867 -0.069645j,
       0.047774+0.j      , -0.002669-0.002333j,  0.006889+0.017726j, -0.021334+0.015868j, -0.001195+0.002602j,  0.014424+0.025481j,  0.006127+0.001767j],
     [-0.016468-0.086672j, -0.017894+0.024044j, -0.100008+0.001044j, -0.012304+0.008907j,  0.032712-0.040006j,  0.00656 -0.007561j, -0.025727-0.00429j , -0.008458+0.01634j ,  0.023206+0.088123j,
      -0.002669+0.002333j,  0.155788+0.j      ,  0.039153-0.008622j, -0.022235+0.016455j,  0.019389-0.010935j,  0.008679+0.035517j, -0.000401+0.006993j],
     [ 0.015002-0.016456j, -0.009242+0.009113j, -0.041196-0.013788j, -0.014615-0.000503j,  0.028404-0.000193j,  0.002035+0.004031j,  0.01078 -0.015412j, -0.001182+0.000863j, -0.041079-0.01863j ,
       0.006889-0.017726j,  0.039153+0.008622j,  0.02955 +0.j      ,  0.004429+0.005112j,  0.007553+0.00339j ,  0.009245+0.013851j,  0.000191+0.005673j],
     [-0.030252+0.020544j,  0.022548-0.006461j,  0.029473+0.023898j,  0.009379-0.001497j,  0.020798-0.00672j ,  0.009555+0.013215j,  0.007845-0.006381j, -0.000061+0.002326j, -0.02004 -0.023296j,
      -0.021334-0.015868j, -0.022235-0.016455j,  0.004429-0.005112j,  0.033644+0.j      , -0.004911+0.002418j, -0.001413-0.02166j , -0.006545-0.000298j],
     [ 0.011294-0.015986j, -0.00949 +0.005239j, -0.029068-0.003006j, -0.00901 +0.004067j,  0.010087-0.006372j,  0.003405-0.001691j, -0.003943-0.013037j, -0.002384-0.00147j , -0.019002+0.017555j,
      -0.001195-0.002602j,  0.019389+0.010935j,  0.007553-0.00339j , -0.004911-0.002418j,  0.008041+0.j      ,  0.007465+0.005605j,  0.003624+0.001535j],
     [-0.002121-0.029806j, -0.005817+0.029118j, -0.056925+0.041351j, -0.015402+0.018162j,  0.006866+0.007065j, -0.00866 +0.003658j,  0.004216-0.016514j,  0.000331-0.007023j, -0.017364-0.013416j,
       0.014424-0.025481j,  0.008679-0.035517j,  0.009245-0.013851j, -0.001413+0.02166j ,  0.007465-0.005605j,  0.042423+0.j      ,  0.011503-0.003335j],
     [ 0.009477-0.006134j, -0.007208+0.006885j, -0.018943+0.005679j, -0.007477+0.005157j, -0.004596-0.000088j, -0.002563-0.003349j, -0.001045-0.006295j,  0.001039-0.003455j, -0.007849+0.00191j ,
       0.006127-0.001767j, -0.000401-0.006993j,  0.000191-0.005673j, -0.006545+0.000298j,  0.003624-0.001535j,  0.011503+0.003335j,  0.005703+0.j      ]]
    )

    # 调用检测程序
    check_bound_entanglement(rho)